In [1]:
#if (!require("BiocManager", quietly = TRUE))
    #install.packages("BiocManager")

#BiocManager::install("MPRAnalyze")
#Format the data into MPRA project
library("MPRAnalyze")
library("BiocParallel")

dropEnhancer <- function(df_RNA){
    row.names(df_RNA) <-df_RNA$enhancer_id	
    df <- df_RNA[ , !(names(df_RNA) %in% c("enhancer_id"))]
    return(df)
    }
    
dropX <- function(df_RNA){
    row.names(df_RNA) <-df_RNA$X
    df <- df_RNA[ , !(names(df_RNA) %in% c("X"))]
    return(df)
    }
    
# Function to load data and convert it to matrix after dropping enhancer
load_and_process <- function(filepath, drop_func) {
  df <- read.csv(filepath, header = TRUE,row.names = 1)
  as.matrix(df)
}

register(MulticoreParam(24))
bpparam <- MulticoreParam(24, log = TRUE, stop.on.error = FALSE)

Load experiment

# Motif Allele Comparison

In [2]:
# Load datasets
df_DNA <- load_and_process("read_counts_R1R2/HEK293T_DNA_matched_barcodes_reshaped_motif.csv", dropEnhancer)
df_RNA <- load_and_process("read_counts_R1R2/HEK293T_RNA_matched_barcodes_reshaped_motif.csv", dropEnhancer)
annot_DNA <- read.csv("annotation_barcodes/mpra3_annot_HEK293T_barcodes_motif.csv",row.names = 1)
df_DNA <- df_DNA[, !grepl("ZC65|ZC66", colnames(df_DNA))]
df_RNA <- df_RNA[, !grepl("ZC65|ZC66", colnames(df_RNA))]
annot_DNA <- annot_DNA[!grepl("ZC65|ZC66", rownames(annot_DNA)), ]

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
                  
#Comparative analysisl
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele+Test,
                          rnaDesign = ~ Motif_Allele,
                          reducedDesign = ~ 1,
                          #mode="scale"
                          )
                      
res <-testLrt(obj)
write.csv(res,"20240719_comparative_HEK293T_motif.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-07-19 15:06:50.949724
Success: TRUE

Task duration:
   user  system elapsed 
  5.075   0.138  27.519 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6289154 335.9   12115641 647.1  7496626 400.4
Vcells 11304860  86.3   18568160 141.7 14920180 113.9

Log messages:
INFO [2024-07-19 15:06:23] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 19
Node: 6
Timestamp: 2024-07-19 15:06:57.890577
Success: TRUE

Task duration:
   user  system elapsed 
 17.741   0.176  34.769 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6290102 336.0   12115641 647.1  7496626 400.4
Vcells 11307670  86.3   18568160 141.7 14920180 113.9

Log messages:
INFO [2024-07-19 15:06:23] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 5
Node: 20
Timestamp: 2024-07-19 15:0

# Allele Only

In [5]:
# Load datasets
df_DNA <- load_and_process("read_counts_R1R2/allele_only/HEK293T_DNA_matched_barcodes_reshaped_allele.csv", dropEnhancer)
df_RNA <- load_and_process("read_counts_R1R2/allele_only/HEK293T_RNA_matched_barcodes_reshaped_allele.csv", dropEnhancer)
annot_DNA <- read.csv("annotation_barcodes/mpra3_annot_HEK293T_barcodes.csv",row.names = 1)
df_DNA <- df_DNA[, !grepl("ZC65|ZC66", colnames(df_DNA))]
df_RNA <- df_RNA[, !grepl("ZC65|ZC66", colnames(df_RNA))]
annot_DNA <- annot_DNA[!grepl("ZC65|ZC66", rownames(annot_DNA)), ]
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

# Define columns to select based on a pattern
total_columns <- nrow(annot_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]
annot_DNA <- annot_DNA[selected_columns, ]#

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
                  
#Comparative analysisl
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele+Test,
                          rnaDesign = ~ Allele_String,
                          reducedDesign = ~ 1,
                          #mode="scale"
                          )
                      
res <-testLrt(obj)
write.csv(res,"20240719_comparative_HEK293T_alleleOnly.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 4
Node: 21
Timestamp: 2024-07-19 15:14:25.715028
Success: TRUE

Task duration:
   user  system elapsed 
 29.971   0.302  30.275 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6354876 339.4   12115641 647.1 10697216 571.3
Vcells 11500666  87.8   22361792 170.7 19841424 151.4

Log messages:
INFO [2024-07-19 15:13:55] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 21
Node: 4
Timestamp: 2024-07-19 15:14:26.595349
Success: TRUE

Task duration:
   user  system elapsed 
 30.128   0.249  30.379 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6354007 339.4   12115641 647.1 10697216 571.3
Vcells 11502559  87.8   22361792 170.7 19841424 151.4

Log messages:
INFO [2024-07-19 15:13:55] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 10
Node: 15
Timestamp: 2024-07-19 15: